In [2]:
import pandas as pd
import numpy as np
import lfp.lfp_analysis.LFP_collection as LFP_collection
import lfp.lfp_analysis.plotting as lfplt
import lfp.lfp_analysis.event_extraction as ee
import pickle
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import ScalarFormatter
from itertools import combinations, permutations
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from importlib import reload
from collections import defaultdict

def unpickle_this(pickle_file):
    """
    Unpickles things
    Args (1):
        file_name: str, pickle filename that already exists and ends with .pkl
    Returns:
        pickled item
    """
    with open(pickle_file, "rb") as file:
        return pickle.load(file)
    


c:\Users\megha\anaconda3\envs\ephys_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

cups_lfp_json = r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Cups (phase 4)\lfp_data\lfp_collection.json"

cups_collection = LFP_collection.LFPCollection.load_collection(cups_lfp_json)


behavior_dicts = unpickle_this('pilot2/cups_phase4/cups_behavior_dicts_by_frame.pkl')

In [3]:
exclusion_dict = {'1.1': [],
                  '1.2': [],
                  '1.3': ['BLA'],
                  '2.1': [],
                  '2.2': ['vHPC'],
                  '2.3': ['mPFC', 'vHPC'],
                  '2.4': ['BLA', 'MD'],
                  '3.1': ['vHPC'],
                  '3.2': [],
                  '3.3': ['MD', 'NAc', 'vHPC'],
                  '4.1': [],
                  '4.4': ['NAc']}
cups_collection.exclude_regions(exclusion_dict)

In [4]:
cups_collection.interpolate()

In [5]:
for recording in cups_collection.recordings:
    subject = str(int(recording.name.split('_')[0])/10)
    recording_pattern = (recording.name.split('_')[0] + '_' +
                         recording.name.split('_')[1] + '_' +
                         recording.name.split('_')[2] + '_' +
                         'aggregated')
    recording.event_dict = behavior_dicts[subject]
    recording.subject = subject
    recording.subject = subject
cups_collection.brain_region_dict


bidict({'mPFC': 0, 'NAc': 1, 'MD': 2, 'BLA': 3, 'vHPC': 4})

In [6]:
def expected_nan_pct(excluded_regions, n_regions=5):
    """
    Calculate expected NaN percentage based on excluded regions.
    
    For power: NaN for each excluded region's channel
    For coherence/granger: NaN for any pair involving an excluded region
    """
    n_excluded = len(excluded_regions)
    
    # Power: one value per region
    power_nan = n_excluded / n_regions * 100
    
    # Coherence/Granger: n×n matrix, diagonal already NaN
    # Count pairs where at least one region is excluded
    total_directed_pairs = n_regions * (n_regions - 1)  # off-diagonal
    # Pairs involving excluded regions (from OR to excluded)
    nan_pairs = n_excluded * (n_regions - 1) * 2 - n_excluded * (n_excluded - 1)
    # Subtract double-counted excluded-to-excluded pairs
    conn_nan = nan_pairs / total_directed_pairs * 100
    
    return power_nan, conn_nan

for recording in cups_collection.recordings:
    n_regions = len(recording.brain_region_dict)
    
    exp_power_nan, exp_conn_nan = expected_nan_pct(
        recording.excluded_regions, 
        n_regions=n_regions
    )
    
    actual_power_nan    = np.mean(np.isnan(recording.power))    * 100
    actual_coherence_nan = np.mean(np.isnan(recording.coherence)) * 100
    actual_granger_nan  = np.mean(np.isnan(recording.granger))  * 100
    
    # Flag subjects where actual NaN exceeds expected
    power_flag    = "⚠️" if actual_power_nan    > exp_power_nan    + 5 else "✓"
    coherence_flag = "⚠️" if actual_coherence_nan > exp_conn_nan     + 5 else "✓"
    granger_flag  = "⚠️" if actual_granger_nan  > exp_conn_nan     + 5 else "✓"
    
    print(
        f"Subject: {recording.subject}  "
        f"Excluded: {recording.excluded_regions}  \n"
        f"  Power:     actual={actual_power_nan:.1f}%  "
        f"expected={exp_power_nan:.1f}%  {power_flag}\n"
        f"  Coherence: actual={actual_coherence_nan:.1f}%  "
        f"expected={exp_conn_nan:.1f}%  {coherence_flag}\n"
        f"  Granger:   actual={actual_granger_nan:.1f}%  "
        f"expected={exp_conn_nan:.1f}%  {granger_flag}\n"
    )

Subject: 1.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 1.3  Excluded: ['BLA']  
  Power:     actual=20.0%  expected=20.0%  ✓
  Coherence: actual=32.0%  expected=40.0%  ✓
  Granger:   actual=32.0%  expected=40.0%  ✓

Subject: 2.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 2.2  Excluded: ['vHPC']  
  Power:     actual=20.0%  expected=20.0%  ✓
  Coherence: actual=32.0%  expected=40.0%  ✓
  Granger:   actual=32.0%  expected=40.0%  ✓

Subject: 2.3  Excluded: ['mPFC', 'vHPC']  
  Power:     actual=40.0%  expected=40.0%  ✓
  Coherence: actual=56.0%  expected=70.0%  ✓
  Granger:   actual=56.0%  expected=70.0%  ✓

Subject: 2.4  Excluded: ['BLA', 'MD']  
  Power:     actual=40.0%  expected=40.0%  ✓
  Coherence: actual=56.0%  expected=70.0%  ✓
  Granger:   actual=56.0%  expec

In [7]:
import itertools
reload(ee)
regions_bidict = cups_collection.brain_region_dict
averages = {}
events = ['cagemate','novel', 'familiar']
baselines = ['cagemate_baseline', 'novel_baseline', 'familiar_baseline']

def create_nested_dict():
    return defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list)))))

averages = create_nested_dict()

for recording in cups_collection.recordings: 
    subject = recording.subject
    
    for mode in ['power', 'coherence', 'granger']:
        regions = list(regions_bidict.keys())
        event_avg_dict = ee.baselined_events(recording, events=events, mode=mode, baseline=baselines)
        event_avg_dict_general_baseline = ee.baselined_events(recording, events=events, mode=mode, baseline='baseline')
        agent_band_dict  = ee.band_calcs(event_avg_dict, outer_dict = 'agent', freq_axis = 1)
        agent_band_dict_gen= ee.band_calcs(event_avg_dict_general_baseline, outer_dict= 'agent', freq_axis = 1)

        for event, bands in agent_band_dict.items():
            for band in bands.keys():
                #print(bands[band][0].shape) #[trials, region]
                band_average = bands[band][0]
                band_average_gen = agent_band_dict_gen[event][band][0]
                
                if mode == 'power':
                    for brain in regions:
                        reg_idx = cups_collection.brain_region_dict[brain]
                        # Store all trials for this region
                        trials_data = [band_average[trial][reg_idx] for trial in range(band_average.shape[0])]
                        trials_data_gen = [band_average_gen[trial][reg_idx] for trial in range(band_average_gen.shape[0])]
                        averages[subject][event][band]['power'][brain] = trials_data
                        averages[subject][event][band]['power_gen_baseline'][brain] = trials_data_gen
                
                elif mode == 'coherence':           
                    for region1, region2 in itertools.combinations(regions, 2):
                        idx1 = cups_collection.brain_region_dict[region1]
                        idx2 = cups_collection.brain_region_dict[region2]
                        region_pair = f"{region1}_{region2}"
                        
                        # Store all trials for this region pair
                        trials_data = [band_average[trial, idx1, idx2] for trial in range(band_average.shape[0])]
                        trials_data_gen = [band_average_gen[trial, idx1, idx2] for trial in range(band_average_gen.shape[0])]
                        
                        averages[subject][event][band]['coherence'][region_pair] = trials_data
                        averages[subject][event][band]['coherence_gen_baseline'][region_pair] = trials_data_gen
                
                elif mode == 'granger':
                    for from_region, to_region in itertools.permutations(regions, 2):
                        from_idx = cups_collection.brain_region_dict[from_region]
                        to_idx = cups_collection.brain_region_dict[to_region]
                        region_pair = f"{from_region}_to_{to_region}"
                        
                        # Store all trials for this directed region pair
                        trials_data = [band_average[trial, to_idx, from_idx] for trial in range(band_average.shape[0])]
                        trials_data_gen = [band_average_gen[trial, to_idx, from_idx] for trial in range(band_average_gen.shape[0])]
                        
                        averages[subject][event][band]['granger'][region_pair] = trials_data
                        averages[subject][event][band]['granger_gen_baseline'][region_pair] = trials_data_gen


def nested_dict_to_long_df(nested_dict):
    """
    Convert nested dictionary to long format DataFrame
    Structure: subject -> event -> band -> metric -> region_or_pair -> [trial_values]
    """
    rows = []
    
    for subject in nested_dict.keys():
        for event in nested_dict[subject].keys():
            for band in nested_dict[subject][event].keys():
                for metric in nested_dict[subject][event][band].keys():
                    for region_or_pair in nested_dict[subject][event][band][metric].keys():
                        trial_values = nested_dict[subject][event][band][metric][region_or_pair]
                        # Create a row for each trial
                        for trial_idx, value in enumerate(trial_values):
                            if 'gen_baseline' in metric:
                                gen_baseline = 1
                            else:
                                gen_baseline = 0
                           
                            row = {
                                'subject': subject,
                                'event': event,
                                'band': band,
                                'metric': metric.replace('_gen_baseline',''),
                                'gen_baseline': gen_baseline,
                                'region_or_pair': region_or_pair,
                                'trial': trial_idx,
                                'value': value
                            }
                            rows.append(row)
    
    return pd.DataFrame(rows)

# Convert to long dataframe
df_long = nested_dict_to_long_df(averages)
df_long
df_long.to_csv("pilot2/cups_phase4/r_stuff/ultra_long_neural_data_cups.csv")

c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:115: RuntimeWarning: Mean of empty slice
  event_snippet = np.nanmean(event_snippet, axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:182: RuntimeWarning: Mean of empty slice
  baseline_recording = np.nanmean(np.array(baseline_averages), axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:332: RuntimeWarning: Mean of empty slice
  band_mean = np.nanmean(arr[tuple(slices)], axis=freq_axis)


In [8]:
import numpy as np
import pandas as pd

rows = []

for subject, events_dict in averages.items():
    for event, bands_dict in events_dict.items():
        for band, metrics_dict in bands_dict.items():
            for metric, regions_dict in metrics_dict.items():
                
                # skip gen_baseline metrics
                if 'gen_baseline' in metric:
                    continue
                
                for region_or_pair, trial_values in regions_dict.items():
                    n_trials = len(trial_values)
                    n_nan    = sum(1 for v in trial_values 
                                   if v is None or 
                                   (isinstance(v, float) and np.isnan(v)))
                    
                    rows.append({
                        'subject'        : subject,
                        'metric'         : metric,
                        'region_or_pair' : region_or_pair,
                        'n_trials'       : n_trials,
                        'n_nan'          : n_nan,
                    })

nan_df = pd.DataFrame(rows)

# Collapse across events and bands — sum trials and nans
summary = (nan_df
    .groupby(['subject', 'metric', 'region_or_pair'])
    .agg(
        total_trials = ('n_trials', 'sum'),
        total_nan    = ('n_nan',    'sum')
    )
    .assign(pct_nan = lambda x: (x.total_nan / x.total_trials * 100).round(1))
    .reset_index()
)

print(summary.to_string(index=False))

# Flag anything with any NaN
print("\nPairs with any NaN:")
print(summary[summary.total_nan > 0].to_string(index=False))

subject    metric region_or_pair  total_trials  total_nan  pct_nan
    1.1 coherence       BLA_vHPC           205          0      0.0
    1.1 coherence         MD_BLA           205          0      0.0
    1.1 coherence        MD_vHPC           205          0      0.0
    1.1 coherence        NAc_BLA           205          0      0.0
    1.1 coherence         NAc_MD           205          0      0.0
    1.1 coherence       NAc_vHPC           205          0      0.0
    1.1 coherence       mPFC_BLA           205          0      0.0
    1.1 coherence        mPFC_MD           205          0      0.0
    1.1 coherence       mPFC_NAc           205          0      0.0
    1.1 coherence      mPFC_vHPC           205          0      0.0
    1.1   granger      BLA_to_MD           205          0      0.0
    1.1   granger     BLA_to_NAc           205          0      0.0
    1.1   granger    BLA_to_mPFC           205          0      0.0
    1.1   granger    BLA_to_vHPC           205          0     

In [1]:
paired_sleap_dicts = unpickle_this('pilot2/only_subects/paired_sleap_dicts.pkl')

NameError: name 'unpickle_this' is not defined

In [9]:
from importlib import reload
reload(LFP_collection)
LFP_collection.LFPCollection.save_to_json(cups_collection, r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Cups (phase 4)\lfp_data_updated", 
                                          notes="Updated with excluded regions, interpolation, and behavior dicts.")